In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
ROOT = Path.cwd().resolve()
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

def performance_stats(df, ret_col="portfolio_return"):
    x = df.copy()
    x = x.dropna(subset=[ret_col])
    r = x[ret_col].fillna(0).to_numpy()
    cum = (1 + pd.Series(r)).cumprod()
    peak = cum.cummax()
    dd = cum / peak - 1
    ann_ret = (cum.iloc[-1] ** (12 / len(r)) - 1) if len(r) > 0 else np.nan
    ann_vol = pd.Series(r).std(ddof=1) * np.sqrt(12) if len(r) > 1 else np.nan
    sharpe = ann_ret / ann_vol if ann_vol not in [0, np.nan] else np.nan
    mdd = dd.min() if len(dd) else np.nan
    return pd.Series({
        "n_months": len(r),
        "cumulative_return": cum.iloc[-1] - 1 if len(cum) else np.nan,
        "annualized_return": ann_ret,
        "annualized_volatility": ann_vol,
        "sharpe_ratio": sharpe,
        "max_drawdown": mdd
    })

a = pd.read_csv(OUT / "backtest_A_core_358.csv")
b = pd.read_csv(OUT / "backtest_B_price_only_500.csv")
c = pd.read_csv(OUT / "backtest_C_overlap.csv")

a_stats = performance_stats(a)
b_stats = performance_stats(b)
c_stats = performance_stats(c)

summary = pd.DataFrame([a_stats, b_stats, c_stats], index=["Model_A", "Model_B", "Model_C"]).reset_index().rename(columns={"index": "model"})
summary.to_csv(OUT / "final_evaluation_summary.csv", index=False)

fm_a = pd.read_csv(OUT / "fama_macbeth_A_summary.csv")
fm_b = pd.read_csv(OUT / "fama_macbeth_B_summary.csv")
fm_c = pd.read_csv(OUT / "fama_macbeth_C_summary.csv")

bayes = pd.read_csv(OUT / "bayes_all_summaries.csv")
bayes_cov = pd.read_csv(OUT / "bayes_coverage.csv")

fm_a["model"] = "A"
fm_b["model"] = "B"
fm_c["model"] = "C"
fm_all = pd.concat([fm_a, fm_b, fm_c], ignore_index=True)
fm_all.to_csv(OUT / "final_fm_summary_all.csv", index=False)

bayes.to_csv(OUT / "final_bayes_summary_all.csv", index=False)
bayes_cov.to_csv(OUT / "final_bayes_coverage.csv", index=False)

comparison = summary.merge(
    pd.DataFrame({
        "model": ["A", "B", "C"],
        "fm_file": ["fama_macbeth_A_summary.csv", "fama_macbeth_B_summary.csv", "fama_macbeth_C_summary.csv"]
    }),
    left_on="model",
    right_on="model",
    how="left"
)
comparison.to_csv(OUT / "project_comparison_table.csv", index=False)

print(summary)
print(fm_all.head())
print(bayes_cov)

     model  n_months  cumulative_return  annualized_return  \
0  Model_A      59.0           0.524112           0.089491   
1  Model_B     136.0          14.279377           0.271979   
2  Model_C      59.0           0.387305           0.068849   

   annualized_volatility  sharpe_ratio  max_drawdown  
0               0.181868      0.492067     -0.267550  
1               0.204560      1.329580     -0.203432  
2               0.176805      0.389406     -0.254624  
           factor  mean_coef    t_stat  n_months model
0           const   0.011245  1.337572        47     A
1     value_score  -0.001883 -0.583992        47     A
2   quality_score   0.006537  0.770644        47     A
3           sue_z  -0.001151 -0.264271        47     A
4  momentum_12m_z   0.008022  1.155534        47     A
              model  n_obs  n_tickers
0        A_core_358   4822        325
1  B_price_only_500  64650        498
2         C_overlap   4822        325
